# 20. Apply a batch of changes to an SCD2 table

**Track:** pyspark | **Difficulty:** hard | **Tags:** scd2, slowly changing dimension, merge, upsert, late-arriving data

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("020-apply-changes-to-scd2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "changes": """
UEFSMRUEFdACFcIBTBUqFQASAACoAQibAwAFAQCLDQgAnw0IAJwNCACVDQgAlw0IAJkNCACYDQgAnQ0IAJ4NCACQDQgAow0IAKQN
CACaDQgAjw0IAJQNCAClDQgAhw0IAIgNCDyKAwAAAAAAAIkDAAAAAAAAFQAVOBU8LBU4FRAVBhUGHBgIpQMAAAAAAAAYCIcDAAAA
AAAAFgAoCKUDAAAAAAAAGAiHAwAAAAAAABERAAAAHGwCAAAAOAEFCSCIQYo5BiW1mGjuwciSnKhRCAAAFQQVqgEVqgFMFRAVABIA
AFXQCQAAAEh5ZGVyYWJhZAYAAABNdW1iYWkHAAAAS29sa2F0YQcAAABDaGVubmFpBAAAAFB1bmUBNSBCZW5nYWx1cnUBNThKYWlw
dXIFAAAARGVsaGkVABUoFSwsFTgVEBUGFQYcNgAoBFB1bmUYCUJlbmdhbHVydRERAAAAFEwCAAAAOAEDCYiGjK3H6N145fcHABUE
FeABFeYBTBU4FQASAABw8G/RTAAA200AAF1NAACATQAA304AAAhNAACDTgAA500AAFZOAABKTAAA4ksAAPdOAAD8TgAAEE8AAERN
AAB9TAAAB00AAAFNAAAETwAAfE4AANVOAAAFTAAADkwAAHFMAADrSwAAqkwAAHFOAAAZTwAAFQAVOBU8LBU4FRAVBhUGHBgEGU8A
ABgE4ksAABYAKAQZTwAAGATiSwAAEREAAAAcbAIAAAA4AQUJIIhBijkoqcWaezDKSau9OOsNAAAVBBlMNQAYBnNjaGVtYRUGABUE
JQIYC2N1c3RvbWVyX2lkABUMJQIYBGNpdHklAEwcAAAAFQIlAhgKY2hhbmdlZF9vbiUMTGwAAAAWOBkcGTwmABwVBBk1AAYQGRgL
Y3VzdG9tZXJfaWQVAhY4FqYEFpwDJuoBJggcGAilAwAAAAAAABgIhwMAAAAAAAAWACgIpQMAAAAAAAAYCIcDAAAAAAAAEREAGSwV
BBUAFQIAFQAVEBUCADwpBhkmADgAAAAmABwVDBk1AAYQGRgEY2l0eRUCFjgWwgIWxgIm7gQmpAMcNgAoBFB1bmUYCUJlbmdhbHVy
dRERABksFQQVABUCABUAFRAVAgA8FuoCGQYZJgA4AAAAJgAcFQIZNQAGEBkYCmNoYW5nZWRfb24VAhY4FpYDFqADJvAHJuoFHBgE
GU8AABgE4ksAABYAKAQZTwAAGATiSwAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmADgAAAAW/gkWOCYIFoIJABkcGAxBUlJPVzpz
Y2hlbWEYzAIvLy8vLy9BQUFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFB
QUFBTUFBQUNBQUFBQVFBQUFBQVFBQUFDYy8vLy9BQUFCQ0JBQUFBQWtBQUFBQkFBQUFBQUFBQUFLQUFBQVkyaGhibWRsWkY5dmJn
QUFBQUFHQUFnQUJnQUdBQUFBQUFBQUFOVC8vLzhBQUFFRkVBQUFBQndBQUFBRUFBQUFBQUFBQUFRQUFBQmphWFI1QUFBQUFBUUFC
QUFFQUFBQUVBQVVBQWdBQmdBSEFBd0FBQUFRQUJBQUFBQUFBQUVDRUFBQUFDUUFBQUFFQUFBQUFBQUFBQXNBQUFCamRYTjBiMjFs
Y2w5cFpBQUlBQXdBQ0FBSEFBZ0FBQUFBQUFBQlFBQUFBQUFBQUFBPQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4x
GTwcAAAcAAAcAAAADQMAAFBBUjE=
""",
    "dim_customer": """
UEFSMRUEFeADFYoCTBU8FQASAADwAQiLAwAFAQCdDQgAnA0IAJoNCACODQgAlw0IAI8NCACJDQgAig0IAI0NCACVDQgAiA0IAJAN
CACSDQgAnw0IAJENCAChDQgAkw0IAJkNCACUDQgAoA0IAJgNCACMDQgAlg0IAIcNCACeDQgAmw0IAKINCDyFAwAAAAAAAIYDAAAA
AAAAFQAVYBVkLBV0FRAVBhUGHBgIogMAAAAAAAAYCIUDAAAAAAAAFgAoCKIDAAAAAAAAGAiFAwAAAAAAABERAAAAMLwCAAAAdAEF
ESCIQYo5CCS1WHMPDsNAlFMcKmur6hSriL0DYpGx1JtTaEvtDAMAAAAVBBWqARWsAUwVEBUAEgAAVcgHAAAAQ2hlbm5haQQAAABQ
dW5lBgAAAE11bWJhaQUAAABEZWxoaQkAAABIeWRlcmFiYWQBM3RLb2xrYXRhCQAAAEJlbmdhbHVydQYAAABKYWlwdXIVABVAFUQs
FXQVEBUGFQYcNgAoBFB1bmUYCUJlbmdhbHVydRERAAAAIHwCAAAAdAEDEUg0Do1UmdHrTRcvNfCmxqvzBDdRFDQAABUEFcgDFdAD
TBVyFQASAADkAfDjMU0AABhMAACyTAAAn0sAAIFMAADsSwAA8UsAAMpNAADNSwAAJUwAAClNAACxTAAAA0wAAD5OAACLTAAAJk0A
AERMAACVTAAAX0wAAGJMAADZTQAA1kwAAMBMAAA4TgAAaEwAAEBNAAAiTQAAek0AAB5NAACNTAAAP0wAAL9NAACCTQAATU4AAJRM
AADaTAAAU04AANFLAACUTQAAmkwAAAhMAAB8TQAA+EwAAIBNAADJSwAAoE0AALFNAADlTAAApkwAALdMAAAKTAAAR00AAOBMAADL
TAAAG0wAAONMAADjTQAAFQAVcBV0LBV0FRAVBhUGHBgEU04AABgEn0sAABYAKARTTgAAGASfSwAAEREAAAA43AIAAAB0AQYRQCAM
RGEcSKIsTOM8UCRNVGVdWKZtXOd9YCiOZGmeaKqubOu+8BvLM13bNw4AAAAAFQQV4AEV5gFMFTgVABIAAHDwb0BNAAAITAAAlEwA
AGJMAABNTgAA1kwAAIJNAAB8TQAAPk4AADFNAACATQAAyk0AALFNAADLTAAAi0wAAFNOAADgTAAAR00AAIFMAABfTAAAoE0AAONN
AAA4TgAAek0AAJVMAACUTQAAv00AANlNAAAVABVGFUosFXQVEBUGFQYcGARTTgAAGAQITAAAFjwoBFNOAAAYBAhMAAAREQAAACOI
CQAAABHsCjJ2LWVcAQUJIIhBijkoqcWaezDKSau9OOsNAAAVABUcFSAsFXQVABUGFQYcGAEBGAEAFgAoAQEYAQAREQAAAA40AgAA
AHQBE/XNidKaowIVBBlsNQAYBnNjaGVtYRUKABUEJQIYC2N1c3RvbWVyX2lkABUMJQIYBGNpdHklAEwcAAAAFQIlAhgKdmFsaWRf
ZnJvbSUMTGwAAAAVAiUCGAh2YWxpZF90byUMTGwAAAAVACUCGAppc19jdXJyZW50ABZ0GRwZXCYAHBUEGTUABhAZGAtjdXN0b21l
cl9pZBUCFnQW3gUWjAQmsgImCBwYCKIDAAAAAAAAGAiFAwAAAAAAABYAKAiiAwAAAAAAABgIhQMAAAAAAAAREQAZLBUEFQAVAgAV
ABUQFQIAPCkGGSYAdAAAACYAHBUMGTUABhAZGARjaXR5FQIWdBbaAhbgAibgBSaUBBw2ACgEUHVuZRgJQmVuZ2FsdXJ1EREAGSwV
BBUAFQIAFQAVEBUCADwW7AUZBhkmAHQAAAAmABwVAhk1AAYQGRgKdmFsaWRfZnJvbRUCFnQWtgUWwgUm5Aom9AYcGARTTgAAGASf
SwAAFgAoBFNOAAAYBJ9LAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAdAAAACYAHBUCGTUABhAZGAh2YWxpZF90bxUCFnQWpAMW
rgMmvA4mtgwcGARTTgAAGAQITAAAFjwoBFNOAAAYBAhMAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSY8OAAAACYAHBUAGSUGABkY
CmlzX2N1cnJlbnQVAhZ0FmIWZibkDzwYAQEYAQAWACgBARgBABERABkcFQAVABUCADwpBhkmAHQAAAAW9BEWdCYIFsIQABkcGAxB
UlJPVzpzY2hlbWEYzAMvLy8vLzFBQkFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlB
QUFBQkFBQUFBVUFBQURrQUFBQXBBQUFBR2dBQUFBMEFBQUFCQUFBQUVELy8vOEFBQUVHRUFBQUFCd0FBQUFFQUFBQUFBQUFBQW9B
QUFCcGMxOWpkWEp5Wlc1MEFBQncvLy8vYlAvLy93QUFBUWdRQUFBQUhBQUFBQVFBQUFBQUFBQUFDQUFBQUhaaGJHbGtYM1J2QUFB
QUFNNy8vLzhBQUFBQW5QLy8vd0FBQVFnUUFBQUFKQUFBQUFRQUFBQUFBQUFBQ2dBQUFIWmhiR2xrWDJaeWIyMEFBQUFBQmdBSUFB
WUFCZ0FBQUFBQUFBRFUvLy8vQUFBQkJSQUFBQUFjQUFBQUJBQUFBQUFBQUFBRUFBQUFZMmwwZVFBQUFBQUVBQVFBQkFBQUFCQUFG
QUFJQUFZQUJ3QU1BQUFBRUFBUUFBQUFBQUFCQWhBQUFBQWtBQUFBQkFBQUFBQUFBQUFMQUFBQVkzVnpkRzl0WlhKZmFXUUFDQUFN
QUFnQUJ3QUlBQUFBQUFBQUFVQUFBQUE9ABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZXBwAABwAABwAABwAABwA
AABcBAAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "020-apply-changes-to-scd2"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

changes = spark.read.parquet((_data_dir / "changes.parquet").resolve().as_posix())
changes.createOrReplaceTempView("changes")
dim_customer = spark.read.parquet((_data_dir / "dim_customer.parquet").resolve().as_posix())
dim_customer.createOrReplaceTempView("dim_customer")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'city': 'a648027fa5fce57a2ea27002cb951c02325a8d9630b2bbe5239bf6791d1bf82e',
                         'customer_id': 'bc5efa0addc9e4a70a54fbff56b5cdeb6ff13ce22badb4fd59b991f27a7d34b4',
                         'is_current': '13040473300758b5e1291d3006aaacffa22944973ba50d2f83c753db4624f497',
                         'valid_from': '9f6b3ce32563a137d1ac564ea7be8eabc233a4b19b414e28259c0093385739e7',
                         'valid_to': '978d2d3565c05d6e23d9438d123f1c2f3437eab5578d4a7370a2d6938a79edf5'},
 'columns': [['city', 'string'],
             ['customer_id', 'bigint'],
             ['is_current', 'boolean'],
             ['valid_from', 'date'],
             ['valid_to', 'date']],
 'fingerprint': 'b1900d9af71580973e3db4917d8fc1cc6b602226b5d27b672fa972eee6037b9c',
 'order_matters': False,
 'row_count': 79}

check = make_check(EXPECTED)

## The situation

`dim_customer` holds the history of each customer's city as a type 2 slowly changing
dimension. Every night a batch of changes arrives from the CRM and has to be applied
to it.

The batch is what the CRM saw, not what the dimension needs. It contains saves that
changed nothing, several changes for the same customer, customers the dimension has
never seen, and changes that reached the feed weeks late.

## Input tables

### `dim_customer`

The dimension before the batch. One row per version of a customer's city.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | The customer. |
| `city` | STRING | The city during this version. Never NULL. |
| `valid_from` | DATE | First day of the version. |
| `valid_to` | DATE | The day the next version started. NULL for the current version. |
| `is_current` | BOOLEAN | True for the customer's latest version. |

The table is well formed: every customer has exactly one current row, one version
ends on the day the next starts, and two consecutive versions never have the same city.

### `changes`

The batch. One row per reported change. A customer has at most one row per date.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | The customer. May be new to the dimension. |
| `city` | STRING | The city reported. Never NULL. |
| `changed_on` | DATE | The date the change took effect. |

## Requirement

Return the complete dimension as it should look after the batch: every existing
version, updated where needed, plus the new versions.

- **Late changes are ignored.** For a customer already in the dimension, a change
  counts only if `changed_on` is after the `valid_from` of the customer's current
  version. A change dated on or before it is discarded. For a customer who is new to
  the dimension, every change counts.
- **Changes are applied in date order.** Go through a customer's accepted changes
  from the earliest `changed_on` to the latest.
- **A change to the same city is not a change.** If the reported city equals the
  city of the version in force at that point, nothing happens.
- **A real change closes one version and opens another.** The version in force gets
  `valid_to` = `changed_on` and `is_current` = false. A new version starts with
  `valid_from` = `changed_on`.
- **A new customer** gets a first version starting at their earliest change.
- After the batch, each customer's latest version has `valid_to` = NULL and
  `is_current` = true. All others have `is_current` = false.
- Customers with no accepted change keep their rows exactly as they are.

## Expected output

One row per version, for all customers.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `city` | STRING | |
| `valid_from` | DATE | |
| `valid_to` | DATE | NULL for the current version. |
| `is_current` | BOOLEAN | |

In [ ]:
changes.show(5)
dim_customer.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `changes`, `dim_customer`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)